# Introduction 
The orgiginal scope of this notebook was to download 100 reviews from 100 games for 10,000 total reviews. That scope was even too large for Kaggle and caused memory overflow errors. Part of the reason for this is the variable length of the steam reviews.

Tokenizing apparently fills up ram very quickly when trying to fine tune a model to your task.

There might be some out of the box sentiment models worth exploring that don't require fine tuning but that's another project for another time.

In [8]:
# For accessing steam game ids
import requests
import pandas as pd
from tqdm import tqdm

# For downloading acutal reviews
import os
from steam.webapi import WebAPI
from dotenv import load_dotenv

In [4]:
# Load Keys
load_dotenv()
STEAM_API_KEY = os.environ.get("STEAM_API_KEY")

# load steam webAPI
api = WebAPI(key=STEAM_API_KEY)

In [18]:
# Get top 100 games by player count from SteamSpy
response = requests.get('https://steamspy.com/api.php?request=top100in2weeks')
top_games = pd.DataFrame(response.json()).T #shortand for JSON transpose rows to cols
top_games = top_games.head(100) #get first 100 rows
top_games['appid'] = top_games['appid'].astype(int) #convert appid to integer
top_games.reset_index(drop=True).head()

,appid,name,developer,publisher,score_rank,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu
0,730,Counter-Strike: Global Offensive,Valve,Valve,,7642084,1173003,0,"100,000,000 .. 200,000,000",31672,877,5137,328,0,0,0,1013936
1,1172470,Apex Legends,Respawn,Electronic Arts,,668053,326926,0,"100,000,000 .. 200,000,000",9726,864,745,285,0,0,0,124262
2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",,1520457,1037487,0,"100,000,000 .. 200,000,000",23286,905,5981,330,0,0,0,314682
3,1623730,Palworld,Pocketpair,Pocketpair,,358266,22443,0,"50,000,000 .. 100,000,000",3603,1104,2114,483,2999,2999,0,18028
4,440,Team Fortress 2,Valve,Valve,,1044264,117208,0,"50,000,000 .. 100,000,000",10283,1170,362,200,0,0,0,43819


In [19]:
top_df = top_games[['appid', 'name']].reset_index(drop=True)
top_df.head()

,appid,name
0,730,Counter-Strike: Global Offensive
1,1172470,Apex Legends
2,578080,PUBG: BATTLEGROUNDS
3,1623730,Palworld
4,440,Team Fortress 2


In [5]:
# Get reviews for a single app
def get_reviews(appid, num_reviews=100, cursor='*'):
    url = f"https://store.steampowered.com/appreviews/{appid}"
    params = {
        'json': 1,
        'num_per_page': num_reviews,
        'cursor': cursor,
        'filter': 'recent',
        'language': 'english'
    }
    r = requests.get(url, params=params)
    if r.status_code == 200:
        data = r.json()
        if 'reviews' in data:
            return { 'cursor': data['cursor'], 'reviews': data['reviews'] }
    return []

In [6]:
# Preview of CS review JSON data
cs_reviews = get_reviews(730, num_reviews=1)
cs_reviews['reviews'][0]

{'recommendationid': '199719695',
 'author': {'steamid': '76561198317761075',
  'num_games_owned': 1,
  'num_reviews': 1,
  'playtime_forever': 4773,
  'playtime_last_two_weeks': 67,
  'playtime_at_review': 4743,
  'last_played': 1752425298},
 'language': 'english',
 'review': 'fix packet loss\r\n',
 'timestamp_created': 1752423430,
 'timestamp_updated': 1752423430,
 'voted_up': False,
 'votes_up': 0,
 'votes_funny': 0,
 'weighted_vote_score': 0.5,
 'comment_count': 0,
 'steam_purchase': True,
 'received_for_free': False,
 'written_during_early_access': False,
 'primarily_steam_deck': False}

In [21]:
all_reviews = []
n_games = 1
to_dl = top_df[:n_games]

for _, row in tqdm(to_dl.iterrows(), total=to_dl.shape[0]):
    appid = row['appid']
    name = row['name']
    response = get_reviews(appid, num_reviews=100)
    for review in response['reviews']:
        all_reviews.append({
            'appid': appid,
            'name': name,
            'review': review['review'],
            'timestamp_created': review['timestamp_created'],
            'voted_up': review['voted_up'],
            'votes_up': review['votes_up'],
            'votes_funny': review['votes_funny'],
            'weighted_vote_score': review['weighted_vote_score'],
        })

reviews_df = pd.DataFrame(all_reviews)
reviews_df.head()

100%|██████████| 1/1 [00:00<00:00,  2.44it/s]


,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
0,730,Counter-Strike: Global Offensive,bad game,1752425700,True,1,0,0.523809552192687988
1,730,Counter-Strike: Global Offensive,:)\r\n,1752425327,True,0,0,0.5
2,730,Counter-Strike: Global Offensive,1337,1752425269,True,0,0,0.5
3,730,Counter-Strike: Global Offensive,what a game wwooowww,1752425237,True,0,0,0.5
4,730,Counter-Strike: Global Offensive,gg,1752425211,True,0,0,0.5


In [28]:
reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   appid                100 non-null    int64  
 1   name                 100 non-null    object 
 2   review               100 non-null    object 
 3   timestamp_created    100 non-null    int64  
 4   voted_up             100 non-null    bool   
 5   votes_up             100 non-null    int64  
 6   votes_funny          100 non-null    int64  
 7   weighted_vote_score  100 non-null    float64
dtypes: bool(1), float64(1), int64(4), object(2)
memory usage: 5.7+ KB


appid: The unique steam id for each game   
name: The unique game name  
review: The corpus of all text in the review  
timestamp_created: Unix time (epoch) in UTC (POSIX TIME) seconds since 01/01/1970  
`voted_up`: The reviewer's thumb up (True) or thumb down (False) Score `(our target)`  
votes_up: Number of people who upvoted the review  
votes_funny: Number of people who thought the vote was funny  
weighted_vote_score: Steam's helpfullness score - between 0 to 1 - where low scores are likely spam - 0.5 is the default  
 - sometimes this comes in as an object - we will convert

In [30]:
reviews_df['weighted_vote_score'] = reviews_df['weighted_vote_score'].astype('float64')
reviews_df.weighted_vote_score.describe()

count    100.000000
mean       0.504032
std        0.009100
min        0.500000
25%        0.500000
50%        0.500000
75%        0.500000
max        0.534368
Name: weighted_vote_score, dtype: float64

Our weighted vote scores are mostly at 0.5 or just slightly above. We'll want to be a bit more selective to just avoid spam.


In [46]:
quality_reviews = []
# Pre-filter Reviews
for _, row in to_dl.iterrows():
    n_quality = 0
    appid = row['appid']
    name = row['name']
    cursor = '*'  # Initial cursor

    while n_quality < 90:
        reviews_data = get_reviews(appid, num_reviews=100, cursor=cursor)
        reviews = reviews_data['reviews']
        cursor = reviews_data.get('cursor')

        if not reviews:
            break

        for review in reviews:
            try:
                is_quality = float(review['weighted_vote_score']) >= 0.52
            except Exception:
                continue
            if is_quality:
                n_quality += 1
                quality_reviews.append({
                    'appid': appid,
                    'name': name,
                    'review': review['review'],
                    'timestamp_created': review['timestamp_created'],
                    'voted_up': review['voted_up'],
                    'votes_up': review['votes_up'],
                    'votes_funny': review['votes_funny'],
                    'weighted_vote_score': review['weighted_vote_score'],
                })

quality_reviews_df = pd.DataFrame(quality_reviews)
quality_reviews_df.head()

,appid,name,review,timestamp_created,voted_up,votes_up,votes_funny,weighted_vote_score
0,730,Counter-Strike: Global Offensive,fire,1752428914,True,1,0,0.523809552192687988
1,730,Counter-Strike: Global Offensive,777,1752428503,True,1,0,0.523809552192687988
2,730,Counter-Strike: Global Offensive,good game. nice to play and many sexy skins,1752427024,True,1,0,0.523809552192687988
3,730,Counter-Strike: Global Offensive,bad game,1752425700,True,1,0,0.523809552192687988
4,730,Counter-Strike: Global Offensive,F?\n,1752421982,True,1,0,0.523809552192687988


In [47]:
quality_reviews_df.to_csv("quality_reviews.csv")

Ok, now we have a reasonable amount of data with some context around the game, the review, and the voted up tag.  

We're going to generate a corpus of input and then split the full data into 3 sets- train, validate and test - in 80:10:10 batches

# Start Here for Clean Analysis


In [48]:
# Start here for clean
import pandas as pd
min_df = pd.read_csv("quality_reviews.csv")

In [49]:
# Clean the punctuation
import re

def cleaned(text):
    return re.sub(r'\W+', '_', text).lower()

In [51]:
# Add input Field
df = min_df.copy().reset_index(drop=True)
df['input'] = 'TEXT1: ' + df.review

# Convert the target to boolean ints
# Our target is currently saved as boolean true false - so let's convert to int
df['voted_up'] = df['voted_up'].astype(int)

display(df['input'].head())
display(df.voted_up.head())

0                                          TEXT1: fire
1                                           TEXT1: 777
2    TEXT1: good game. nice to play and many sexy s...
3                                      TEXT1: bad game
4                                          TEXT1: F?\n
Name: input, dtype: object

0    1
1    1
2    1
3    1
4    1
Name: voted_up, dtype: int64

In [52]:
# Now let's get experience with datasets ( required for hugging face transformers )
from datasets import Dataset,DatasetDict

# Select the columns we want to keep for the dataset/prediction purposes
columns = ['input', 'voted_up']
df = df[columns].copy()

ds = Dataset.from_pandas(df)

In [53]:
ds

Dataset({
    features: ['input', 'voted_up'],
    num_rows: 90
})

In [54]:
# We're going to create todenizers using deberta
model_name = 'microsoft/deberta-v3-small'

from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokz = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

d:\Users\mores\miniconda3\envs\kaggle\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mores\.cache\huggingface\hub\models--microsoft--deberta-v3-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

d:\Users\mores\miniconda3\envs\kaggle\Lib\site-packages\transformers\convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


The warning is only an issue because we're using noisy or multilingual data - the reviews, even when marked english, have all kinds of randomness.

Our options are to ignore (if we're ok with slighly less flexible outcomes) - instead of <unk> the characters would be byte level split.  
Or we can use something like..  
> from transformers import T5Tokenizer  
> tokenizer = T5Tokenizer.from_pretrained("t5-base", use_fast=False)  

In [55]:
# Test out the tokenizer with some basic text
tokz.tokenize("TEXT1: An Hello, I'm new at this and learning is fun!")

['▁TEXT',
 '1',
 ':',
 '▁An',
 '▁Hello',
 ',',
 '▁I',
 "'",
 'm',
 '▁new',
 '▁at',
 '▁this',
 '▁and',
 '▁learning',
 '▁is',
 '▁fun',
 '!']

In [56]:
# Vs some other text in the head of an earlier preview
tokz.tokenize("Tässä pelissä on intensiivistä väkivaltaa")

['▁T',
 'ä',
 's',
 's',
 'ä',
 '▁pe',
 'liss',
 'ä',
 '▁on',
 '▁in',
 't',
 'ensi',
 'ivist',
 'ä',
 '▁vä',
 'k',
 'ival',
 'ta',
 'a']

The tokenizer works fine on normal text but not fine on other text

In [57]:
# simple function to tokenize our
def tok_func(x): return tokz(x["input"])

In [58]:
# Parallel for every row in ds with map
tok_ds = ds.map(tok_func, batched=True)

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

In [59]:
row = tok_ds[0]
row['input'], row['input_ids']

('TEXT1: fire', [1, 54453, 435, 294, 1351, 2])

These ids are a list of vocab in the tokenizer with a unique int for every string

In [67]:
# See the int above
tokz.vocab['text']

12948

In [68]:
# Transformers needs a column called labels
tok_ds = tok_ds.rename_columns({'voted_up':'labels'})

In [69]:
tok_ds

Dataset({
    features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 90
})

`tok_ds` is now ready for splitting

In [70]:
# Step 1: Train/Test Split (e.g., 80% train, 20% temp)
train_test = tok_ds.train_test_split(test_size=0.2, seed=42)

# Step 2: Split test portion into validation and test (e.g., 50/50 of the 20%)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

train_ds = train_test['train']
test_ds = val_test['train']
eval_ds = val_test['test']

dds = DatasetDict({
    'train': train_ds,
    'test': test_ds,
    'eval': eval_ds
})

In [71]:
dds

DatasetDict({
    train: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 72
    })
    test: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9
    })
    eval: Dataset({
        features: ['input', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9
    })
})

In [72]:
# To train our model in transformers we need to more modules
from transformers import TrainingArguments, Trainer

In [ ]:
bs = 16 # batch size
epochs = 4
lr = 8e-5 # small learning rate, halve the size if too far

In [74]:
import transformers
from transformers import TrainingArguments
print(transformers.__version__)

4.51.3


In [85]:
args = TrainingArguments(
    'outputs',
    learning_rate=lr,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=False,
    gradient_checkpointing=False,
    eval_strategy="epoch",
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=2,
    num_train_epochs=epochs,
    weight_decay=0.01,
    report_to='none',
    no_cuda=True,    # This is the correct parameter to force CPU usage
    dataloader_num_workers=0  # Helps prevent memory issues
)

d:\Users\mores\miniconda3\envs\kaggle\Lib\site-packages\transformers\training_args.py:1595: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.26.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=0.26.0'`

In [36]:
# Define how to compute the metric or target by SGD
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

In [40]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Slow things down but reduce memory usage
model.gradient_checkpointing_enable()

trainer = Trainer(
    model, 
    args, 
    train_dataset=dds['train'], 
    eval_dataset=dds['test'],
    tokenizer=tokz, 
    compute_metrics=compute_metrics
)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_35/165106228.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Not Enough Memory to continue on Kaggle
For whatever reason - running this fills up the available GPU memory very quickly.  

Probably there is some type of smaller process or batch process that would help process, then clear, then next batch the request.

In [41]:
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:
# Get raw logits
raw_preds = trainer.predict(eval_ds)

# Convert logits to class predictions (0 or 1)
preds = np.argmax(raw_preds.predictions, axis=1)

# If needed, also get ground truth labels
true_labels = raw_preds.label_ids

# Optionally inspect
print(preds[:10])